# tp-18 特徴量グループごとの階層スタッキング（どの階層でどの特徴量を使うか）

**問い**: 今は全特徴量（約 250 列）を同じ段で 1 つのモデルに入れ、出力のところでだけアンサンブルしている。レーティング（能力）・脚質/ペース・適性・状態・関係者の特徴量を**役割ごとに別のモデル（L1）に分け、L2 で組み合わせる**設計のほうが良いか。また、L2 に「生の特徴量の近道（skip connection）」を足す価値がどのグループにあるか。

## グループ（`src/pipeline/models/feature_groups.py`）
| グループ | 中身 |
|----------|------|
| RATING | Elo・相手関係・残差（F3, C）、標準条件レーティング（H）、前段モデル 3 列、馬の通算成績 |
| STYLE | 上がり・脚質（A）、展開・ペースの相性（F2, G）、安定度（E） |
| APTITUDE | 得意な舞台（B）、条件類似度（F1, F5）、血統×条件、種牡馬の傾向 |
| CONDITION | 休養・体重（D）、追い切り評価・厩舎コメント、斤量・年齢・性別、当日の馬場傾向 |
| CONNECTIONS | 騎手・調教師の通算成績 |
| CONTEXT | レース条件・枠番（**全グループのモデルに共通で渡す**。分けると相互作用が読めないため） |

## 比較（try both）
- 基準 A: 全特徴量を 1 つの LightGBM に入れた版 / 基準 B: アンサンブル v2（採用中）
- L1 = グループ別 LightGBM（各 CONTEXT 付き）→ L2: ロジスティック / 条件付きロジット / 浅い LightGBM / 浅い LightGBM + CONTEXT / + 全特徴モデルの出力 / 小さな NN
- 近道: L2 の LightGBM に、各グループの生の特徴量を 1 つずつ足した版（どの階層に置くと得か）
- グループ抜き: L2 から 1 グループずつ外した版（各グループの寄与）
- L3: v2 とグループ L2 の logit 平均（階層が v2 に無い情報を足すか）

L2 は tp-05m と同じ leave-one-year-out（2022〜2024 の OOF 上）。採用基準 = 勝率 LogLoss の差の 95% 信頼区間の上限 < 0、効果 ≥ 0.0002、連対・複勝が 0.0002 を超えて悪化しない（基準 B = v2 に対して）。test 2025 は使わない。

In [ ]:
import os, sys
from pathlib import Path


def _find_root():
    p = Path.cwd().resolve()
    for q in [p, *p.parents]:
        if (q / "AGENTS.md").exists() and (q / "src").is_dir():
            return q
    raise RuntimeError("リポジトリルートが見つかりません")


REPO_ROOT = _find_root()
os.chdir(REPO_ROOT)
os.environ.setdefault("POLARS_MAX_THREADS", "4")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
from src.utils.project_env import load_project_dotenv
load_project_dotenv()

In [ ]:
import json, subprocess, gc, time
import numpy as np
import pandas as pd
import mlflow

from src.pipeline.mlflow.train_utils import require_tracking_server
from src.pipeline.models import ensemble as en, stacking as st, tp05_data, win_models as wm, zoo, zoo_runner, feature_groups as fg
from src.pipeline.models.compute import detect_compute

WORK = REPO_ROOT / "data/local/modeling/training_pipeline"
OOF_DIR = WORK / "oof/model_oof"
OUT = WORK / "improve_h"
OUT.mkdir(parents=True, exist_ok=True)
THREADS = int(os.environ.get("TP18_THREADS", detect_compute(reserve_cores=6).threads))
SEEDS = [0, 1]
SMOKE = os.environ.get("TP18_SMOKE") == "1"
mlflow.set_experiment("keiba-tp-hierarchical-stacking")
require_tracking_server()
GIT_COMMIT = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True, cwd=REPO_ROOT).stdout.strip()
D = tp05_data.load_dataset(WORK)
df, meta, folds = D["df"], D["meta"], D["folds"]
EP, EXTRAS, PED = tp05_data.load_extras_ped(df, WORK)
X0 = pd.concat([df[D["FEATS"]["base+S+C1"]], EP], axis=1)
best = json.loads((WORK / "hpo/best_params.json").read_text(encoding="utf-8"))["best"]
LP = zoo.tuned_params("lgb", best["lgb"]["params"])
fam = json.loads((WORK / "features/feature_families.json").read_text(encoding="utf-8"))
GR = fg.assign_groups(list(X0.columns), fam, tp05_data.STAGE_COLS, EXTRAS, PED)
CTX = [c for c in fg.CONTEXT if c in X0.columns]
assert sorted(c for v in GR.values() for c in v) == sorted(c for c in X0.columns if c not in CTX)
print({g: len(v) for g, v in GR.items()}, "| CONTEXT", len(CTX), "| 合計", X0.shape[1])
if SMOKE:
    folds = folds[:2]
    SEEDS = [0]

## L1: グループ別 LightGBM（CONTEXT 付き）と全特徴モデル

In [ ]:
key = ["race_id", "horse_id"]
L1 = {}
for g, cols in GR.items():
    t0 = time.time()
    res, oof = zoo_runner.run_oof(f"lgb_grp_{g}", "lgb_binary", X0[CTX + cols], meta, folds, SEEDS, THREADS, params=dict(LP))
    L1[g] = oof
    print(f"{g}: {len(cols)} 列 win {res['logloss_win'].mean():.5f} top2 {res['logloss_top2'].mean():.5f} top3 {res['logloss_top3'].mean():.5f} ({time.time() - t0:.0f}s)", flush=True)
t0 = time.time()
resA, L1["ALL"] = zoo_runner.run_oof("lgb_grp_ALL", "lgb_binary", X0, meta, folds, SEEDS, THREADS, params=dict(LP))
print(f"ALL（基準 A）: win {resA['logloss_win'].mean():.5f} top2 {resA['logloss_top2'].mean():.5f} top3 {resA['logloss_top3'].mean():.5f} ({time.time() - t0:.0f}s)", flush=True)
resC, L1["CONTEXT"] = zoo_runner.run_oof("lgb_grp_CONTEXT", "lgb_binary", X0[CTX], meta, folds, SEEDS, THREADS, params=dict(LP))
print(f"CONTEXT のみ: win {resC['logloss_win'].mean():.5f}", flush=True)

In [ ]:
TOT = st.TOT
base = meta[key + ["finish_position", "year"]].copy()
for g, o in L1.items():
    o = o[key + ["p_win", "p_top2", "p_top3"]].rename(columns={f"p_{t}": f"{g}__{t}" for t in TOT})
    base = base.merge(o, on=key, how="inner")
v2 = pd.read_parquet(OOF_DIR / "ensemble_v2.parquet")[key + ["p_win", "p_top2", "p_top3"]]
v2oof = base[key].merge(v2, on=key, how="left")
race, years, fin = base["race_id"].to_numpy(), base["year"].to_numpy(), base["finish_position"].to_numpy()
Y = {"win": (fin == 1).astype(float), "top2": (fin <= 2).astype(float), "top3": (fin <= 3).astype(float)}
V2P = {t: v2oof[f"p_{t}"].to_numpy() for t in TOT}
ALLP = {t: base[f"ALL__{t}"].to_numpy() for t in TOT}
GN = list(GR)
print(f"行 {len(base):,}", "| v2 の欠損", int(v2oof['p_win'].isna().sum()))
print({t: round(en.ll(V2P[t], Y[t]), 5) for t in TOT}, "<- v2 |", {t: round(en.ll(ALLP[t], Y[t]), 5) for t in TOT}, "<- 全特徴 1 モデル")


def cmp_vs(P, ref, name, min_effect=2e-4, tol=2e-4):
    out = {"variant": name}
    for t in TOT:
        b = wm.paired_bootstrap_diff(wm.per_race_loss(race, ref[t], Y[t]), wm.per_race_loss(race, P[t], Y[t]), n_boot=1000)
        out[f"{t}_ref"], out[f"{t}_new"], out[f"Δ{t}"], out[f"{t}_hi"] = en.ll(ref[t], Y[t]), en.ll(P[t], Y[t]), b["diff"], b["hi"]
    out["採用"] = bool(out["win_hi"] < 0 and out["Δwin"] <= -min_effect and out["Δtop2"] <= tol and out["Δtop3"] <= tol)
    return out

## L2: グループの出力を組み合わせる

In [ ]:
def zt_of(names):
    return {t: en.logit(np.clip(np.stack([base[f"{n}__{t}"].to_numpy() for n in names], axis=1), 1e-6, 1 - 1e-6)) for t in TOT}


def meta_F(names, extra=None):
    Zt = zt_of(names)
    mean_all = {t: Zt[t].mean(1) for t in TOT}
    F = {}
    for t in TOT:
        f = st.meta_features(Zt[t], race, names)
        for u in TOT:
            if u != t:
                f[f"zmean_{u}"] = mean_all[u]
        if extra is not None:
            f = pd.concat([f, extra.reset_index(drop=True)], axis=1)
        F[t] = f
    return Zt, F


def codes(frame):
    o = frame.copy()
    for c in o.columns:
        if str(o[c].dtype) in ("category", "object"):
            o[c] = pd.Categorical(o[c]).codes.astype("float32")
        else:
            o[c] = o[c].astype("float32")
    return o


aligned = base[key].merge(pd.concat([df[key], X0], axis=1), on=key, how="left")
CTXF = codes(aligned[CTX])
RAWF = {g: codes(aligned[cols]) for g, cols in GR.items()}
ROWS_V2, ROWS_A, STORE = [], [], {}


def record(P, name):
    STORE[name] = P
    ROWS_V2.append(cmp_vs(P, V2P, name))
    ROWS_A.append(cmp_vs(P, ALLP, name))
    r = ROWS_V2[-1]
    print(f"{name}: win Δ(v2) {r['Δwin']:+.5f} (上限 {r['win_hi']:+.5f}) top2 {r['Δtop2']:+.5f} top3 {r['Δtop3']:+.5f} | 採用={r['採用']}", flush=True)


t0 = time.time()
Zt, F = meta_F(GN)
# (a) ロジスティック
La = {t: en.loyo_eval("stack_lr", en.sigmoid(Zt[t]), years, race, Y[t], TOT[t])[1] for t in TOT}
record(La, "L2 ロジスティック（グループ出力のみ）")
# (b) 条件付きロジット
Lb = st.loyo_clogit_meta(Zt, Y, years, race)
record(Lb, "L2 条件付きロジット")
# (c) 浅い LightGBM
Lc = st.loyo_lgb_meta({t: F[t].to_numpy() for t in TOT}, Y, years, race, threads=THREADS)
record(Lc, "L2 LightGBM")
# (d) LightGBM + CONTEXT
_, Fd = meta_F(GN, CTXF)
Ld = st.loyo_lgb_meta({t: Fd[t].to_numpy() for t in TOT}, Y, years, race, threads=THREADS)
record(Ld, "L2 LightGBM + CONTEXT")
# (e) LightGBM + CONTEXT + 全特徴モデルの出力
_, Fe = meta_F(GN + ["ALL"], CTXF)
Le = st.loyo_lgb_meta({t: Fe[t].to_numpy() for t in TOT}, Y, years, race, threads=THREADS)
record(Le, "L2 LightGBM + CONTEXT + 全特徴モデル")
# (f) 小さな NN
Zt_f = zt_of(GN)
Fall = pd.concat([st.meta_features(Zt_f[t], race, GN, with_context=(t == "win")).add_suffix(f"__{t}") for t in TOT], axis=1)
Lf = st.loyo_nn_meta(Fall, Y, years, race)
record(Lf, "L2 NN")
print(f"L2 基本 {time.time() - t0:.0f}s")

## 近道（skip connection）: L2 に各グループの生の特徴量を足す / L2 から 1 グループを外す

In [ ]:
for g in GN:
    _, Fg = meta_F(GN, pd.concat([CTXF, RAWF[g].add_prefix("raw_")], axis=1))
    P = st.loyo_lgb_meta({t: Fg[t].to_numpy() for t in TOT}, Y, years, race, threads=THREADS)
    record(P, f"近道: L2 LightGBM + CONTEXT + {g} の生特徴")
for g in GN:
    names = [n for n in GN if n != g]
    _, Fo = meta_F(names, CTXF)
    P = st.loyo_lgb_meta({t: Fo[t].to_numpy() for t in TOT}, Y, years, race, threads=THREADS)
    record(P, f"グループ抜き: {g} を外す")

## L3: v2 との併用

In [ ]:
cands = ["L2 LightGBM + CONTEXT", "L2 LightGBM + CONTEXT + 全特徴モデル", "L2 条件付きロジット", "L2 NN"]
for c in cands:
    record({t: st.logit_mean([V2P[t], STORE[c][t]], race, TOT[t]) for t in TOT}, f"L3 = v2 + {c}")
record({t: st.logit_mean([V2P[t]] + [STORE[c][t] for c in cands], race, TOT[t]) for t in TOT}, "L3 = v2 + 全 L2")
R = pd.DataFrame(ROWS_V2).set_index("variant")
RA = pd.DataFrame(ROWS_A).set_index("variant")
R.to_csv(OUT / "hier_vs_v2.csv"); RA.to_csv(OUT / "hier_vs_all_single.csv")
display(R[["win_ref", "win_new", "Δwin", "win_hi", "Δtop2", "Δtop3", "採用"]].round(5))
print("--- 基準 A（全特徴 1 モデル）に対して ---")
display(RA[["win_ref", "win_new", "Δwin", "win_hi", "Δtop2", "Δtop3", "採用"]].round(5))

In [ ]:
# グループ別 L1 の単独の精度と、グループの寄与
single = pd.DataFrame({g: {f"{t}": en.ll(base[f"{g}__{t}"].to_numpy(), Y[t]) for t in TOT} for g in list(L1)}).T
display(single.round(5))
best_name = R.sort_values("Δwin").index[0]
print("v2 に対して勝率で最良:", best_name, {k: (round(float(R.loc[best_name, k]), 5) if k != "採用" else bool(R.loc[best_name, k])) for k in ("Δwin", "win_hi", "Δtop2", "Δtop3", "採用")})
adopted = list(R.index[R["採用"]])
for nm in adopted:
    o = base[key + ["year"]].copy()
    for t in TOT:
        o[f"p_{t}"] = STORE[nm][t]
    o.to_parquet(OOF_DIR / ("stackh_" + "".join(ch if ch.isascii() and ch.isalnum() else "_" for ch in nm)[:60] + ".parquet"), index=False)
(OUT / "report.json").write_text(json.dumps({"git_commit": GIT_COMMIT, "groups": {g: len(v) for g, v in GR.items()}, "results_vs_v2": R.round(6).reset_index().to_dict("records"),
                                             "results_vs_all": RA.round(6).reset_index().to_dict("records"), "single": single.round(6).reset_index().to_dict("records"), "best": best_name, "adopted": adopted},
                                            ensure_ascii=False, indent=1, default=str), encoding="utf-8")
with mlflow.start_run(run_name="tp-18-hierarchical-stacking", tags={"notebook": "tp-18", "git_commit": GIT_COMMIT}):
    for i_, n in enumerate(R.index):
        mlflow.log_metrics({f"delta_win_v2_{i_:02d}": float(R.loc[n, "Δwin"])})
    mlflow.log_dict({f"{i_:02d}": n for i_, n in enumerate(R.index)}, "improve_h/metric_index.json")
    for f_ in ("hier_vs_v2.csv", "hier_vs_all_single.csv", "report.json"):
        mlflow.log_artifact(str(OUT / f_), "improve_h")